# Gradient et Hessienne de machines simples

Ce notebook accompagne les exercices 2.1 et 2.2 du cours. Le calcul mathématique est fait dans les espaces de matrices et de vecteurs du cours ; JAX sert ensuite à vérifier le gradient et surtout l'action de la Hessienne sur une direction.

Nous ne formerons pas une grande matrice hessienne. La transformation `jax.jvp` permet de calculer directement
$$
\nabla^2\mathcal L(p)\,\delta p
=D(\nabla\mathcal L)(p)[\delta p].
$$

## Exercices

1. [Exercice 2.1 — Hessienne de la régression linéaire](#exercice-2-1)
2. [Exercice 2.2 — Gradient et Hessienne d'un perceptron](#exercice-2-2)

In [2]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt

print(f"JAX {jax.__version__}")

JAX 0.11.1


<a id="exercice-2-1"></a>
## Exercice 2.1 — Hessienne de la régression linéaire

Pour $A\in\mathbb R^{m\times n}$ et $b\in\mathbb R^m$, on considère
$$
\mathcal L(A,b)=\frac12\sum_{(x,z)\in\mathcal D}|Ax+b-z|^2.
$$

Montrer que la Hessienne est indépendante de $(A,b)$ et que son action sur $(\delta A,\delta b)$ est
$$
\nabla^2\mathcal L(A,b)(\delta A,\delta b)
=\left(
\sum_{(x,z)\in\mathcal D}(\delta Ax+\delta b)x^\top,
\sum_{(x,z)\in\mathcal D}(\delta Ax+\delta b)
\right).
$$

Établir ensuite
$$
\left\langle\nabla^2\mathcal L(A,b)(\delta A,\delta b),
(\delta A,\delta b)\right\rangle
=\sum_{(x,z)\in\mathcal D}|\delta Ax+\delta b|^2
$$
et en déduire la convexité de $\mathcal L$.

### Vérification avec JAX

Les données sont rangées par lignes : `x` a la forme `(n_data, n)` et la machine calcule donc `x @ A.T + b`. Les paramètres et leur perturbation sont deux dictionnaires de même structure.

In [3]:
cle = jax.random.key(2026)
cle_x, cle_z, cle_A, cle_b, cle_dA, cle_db = jax.random.split(cle, 6)
n_data, n, m = 8, 3, 2

x = jax.random.normal(cle_x, (n_data, n))
z = jax.random.normal(cle_z, (n_data, m))
parametres = {
    "A": jax.random.normal(cle_A, (m, n)),
    "b": jax.random.normal(cle_b, (m,)),
}
direction = {
    "A": jax.random.normal(cle_dA, (m, n)),
    "b": jax.random.normal(cle_db, (m,)),
}

def cout_lineaire(parametres, x, z):
    residu = x @ parametres["A"].T + parametres["b"] - z
    return 0.5 * jnp.sum(residu**2)

Calculez avec JAX l'action de la Hessienne sur `direction`, puis construisez la même action à partir de la formule du cours. Vérifiez enfin la formule de la forme quadratique.

In [ ]:
# À compléter.

### Question complémentaire

La positivité prouve la convexité, mais pas toujours la stricte convexité. Construisez un ensemble de données pour lequel une direction non nulle vérifie $\delta Ax+\delta b=0$ pour toutes les entrées observées. Que devient alors la forme quadratique ?

In [ ]:
# À compléter.

<a id="exercice-2-2"></a>
## Exercice 2.2 — Gradient et Hessienne d'un perceptron

On considère
$$
\Phi(x,W,b)=\sigma(Wx+b)
$$
et
$$
\mathcal L(W,b)=\frac12\sum_{(x,z)\in\mathcal D}
|\sigma(Wx+b)-z|^2.
$$

Avec $a_x=Wx+b$ et $r_{x,z}=\sigma(a_x)-z$, établir les formules du gradient données dans le cours. Pour une direction $(\delta W,\delta b)$, poser
$$
v_x=\delta Wx+\delta b,
\qquad
\gamma_{x,z}=\sigma'(a_x)\odot\sigma'(a_x)
+r_{x,z}\odot\sigma''(a_x),
$$
puis établir
$$
\nabla^2\mathcal L(W,b)(\delta W,\delta b)
=\left(
\sum (\gamma_{x,z}\odot v_x)x^\top,
\sum \gamma_{x,z}\odot v_x
\right).
$$

### Vérification pour $\sigma=\tanh$

Pour `tanh`,
$$
\sigma'(t)=1-\tanh^2(t),
\qquad
\sigma''(t)=-2\tanh(t)(1-\tanh^2(t)).
$$

Implémentez les formules du gradient et de l'action de la Hessienne, puis comparez-les aux résultats de JAX.

In [6]:
parametres_perceptron = {
    "W": jax.random.normal(cle_A, (m, n)),
    "b": jax.random.normal(cle_b, (m,)),
}
direction_perceptron = {
    "W": jax.random.normal(cle_dA, (m, n)),
    "b": jax.random.normal(cle_db, (m,)),
}

def cout_perceptron(parametres, x, z):
    a = x @ parametres["W"].T + parametres["b"]
    residu = jnp.tanh(a) - z
    return 0.5 * jnp.sum(residu**2)

In [ ]:
# À compléter.

### Non-convexité

Dans le cas scalaire $x=z=0$ et $\sigma=\tanh$, la perte ne dépend que de $b$ :
$$
\mathcal L(b)=\frac12\tanh^2(b).
$$

Calculez sa dérivée seconde et trouvez une région où elle est négative.

In [ ]:
# À compléter.

## Bilan

Pour une régression linéaire, la Hessienne est positive et indépendante des paramètres. Pour un perceptron, le terme contenant le résidu et $\sigma''$ peut rendre la Hessienne indéfinie : la fonction objectif n'est donc pas nécessairement convexe.

Dans les deux cas, `jax.jvp(jax.grad(...))` calcule directement l'action de la Hessienne. Cette représentation est plus naturelle et souvent beaucoup moins coûteuse que la construction de toute la matrice hessienne.